# Q4 · LLM extraction from ISW Iran Updates

In [1]:
%load_ext autoreload
%autoreload 2
import json, logging, os, sys
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
from IPython.display import display

logging.basicConfig(level=logging.WARNING, force=True)
pio.renderers.default = "plotly_mimetype+notebook_connected"
pd.set_option("display.width", 160, "display.max_colwidth", 120)

import schema as sc
import isw_pipeline as pp
from examples import GOLD
sys.path.insert(0, "../Q1_Q2"); import charts as ch

## Design drivers
- ISW rarely mentions prices → extract facts, map to inflation channels.
- Hormuz moved in stages → ordered status ladder.
- ISW grades claims → confidence + source fields.
- ISW blocks scripts → pluggable source.

## Architecture
```
ArticleSource    archive / direct / manual upload
   ↓
ISWParser        HTML → text
   ↓
BriefExtractor   PromptBuilder + LLMClient
   ↓             invalid JSON → retry
   ↓             quote not in article → drop
BriefRepository  one JSON per report
   ↓
BriefAnalyzer    timeline · Hormuz state · diffs
```

## Schema
```
ConflictEvent    date, actor, location,
                 confidence, sources, quote
 ├ Strike · MaritimeIncident
 ├ InfrastructureAttack · LeadershipTargeted
 └ Diplomacy · NegativeObservation
DailyBrief       events + Hormuz status + trend
                 → channel_scores, diff()
```
Enums: `InflationChannel` · `ChokepointStatus` · `Confidence` · `EvidenceType` · `Actor` · `AssetType`

## OOP principles
| Principle | Where |
|---|---|
| Abstraction | `ArticleSource`, `LLMClient`, `BriefRepository` are abstract interfaces |
| Encapsulation | Validation lives in the models (e.g. "confirmed" needs hard evidence) |
| Inheritance | Six event types extend `ConflictEvent`; sources share `ArticleSource.get()` |
| Polymorphism | Each event type scores itself; the brief just loops over events |
| Composition | `ExtractionPipeline` = source + extractor + repository |
| SOLID | One job per class; new source or event type = new subclass; LLM injected (`ReplayClient` for tests) |

## Demo: first weeks of the war
Two injected faults: invalid answer (28 Feb), made-up event (2 Mar).

In [2]:
slugs = list(GOLD)
answers = {s: b.model_dump_json() for s, b in GOLD.items()}

bad = json.loads(answers[slugs[0]])                         # fault 1: "confirmed" on a social-media source
bad["events"][0]["confidence"] = 6
bad["events"][0]["sources"] = [{"outlet": "X/Twitter OSINT account", "evidence": "social_media"}]
class FlakyReplay(pp.ReplayClient):
    def __init__(self, answers, first_bad):
        super().__init__(answers); self.first_bad, self.used = first_bad, set()
    def complete(self, system, user, schema):
        slug = user.split("SLUG: ")[1].split()[0]
        if slug in self.first_bad and slug not in self.used:
            self.used.add(slug); return json.dumps(self.first_bad[slug])
        return super().complete(system, user, schema)

mar2 = json.loads(answers[slugs[2]])                        # fault 2: quote that is not in the article
mar2["events"].append({"kind": "infrastructure_attack", "summary": "Kharg Island export terminal destroyed.",
                       "initiator": "israel", "confidence": 4, "sources": [], "damage": "destroyed",
                       "asset": {"name": "Kharg Island", "asset_type": "export_terminal"},
                       "evidence_quote": "Israel destroyed the Kharg Island oil export terminal on March 2."})
answers[slugs[2]] = json.dumps(mar2)

pipe = pp.ExtractionPipeline(source=pp.InternetArchiveSource(),
                             extractor=pp.BriefExtractor(FlakyReplay(answers, {slugs[0]: bad}),
                                                         pp.PromptBuilder(few_shot=[GOLD[slugs[1]]])),
                             repo=pp.JsonFileRepository())
results = pipe.run(slugs)
display(pd.DataFrame([{"report": f"{r.article.report_date:%d %b} {r.article.report_type.value.replace('_', ' ')}",
                       "attempts": r.attempts, "events kept": len(r.brief.events) if r.brief else 0,
                       "dropped": "; ".join(r.dropped_events) or "—"} for r in results]).set_index("report"))

,attempts,events kept,dropped
report,,,
28 Feb special,2,2,—
28 Feb evening special,1,1,—
02 Mar evening special,1,3,Kharg Island export terminal destroyed.
04 Mar evening special,1,1,—
16 Mar evening special,1,2,—


In [3]:
an = pp.BriefAnalyzer([r.brief for r in results if r.brief])
tl, state = an.timeline(), an.chokepoint_state()
chan_cols = [c for c in tl.columns if c not in ("events", "trend")]
palette = ["#d95926", "#3987e5", "#199e70", "#c98500", "#9085e9", "#e66767", "#d55181", "#8a94a6"]
fig = make_subplots(rows=2, cols=1, shared_xaxes=True, row_heights=[0.62, 0.38], vertical_spacing=0.1,
                    subplot_titles=("Signal by inflation channel (news in each report)", "Strait of Hormuz status"))
for c, col in zip(chan_cols, palette):
    fig.add_trace(go.Bar(x=tl.index, y=tl[c], name=c.replace("_", " "), marker_color=col,
                         hovertemplate="%{y:+.2f}<extra>" + c + "</extra>"), 1, 1)
fig.add_trace(go.Scatter(x=state.index, y=state.map(lambda s: sc.ChokepointStatus[s].value), mode="lines+markers+text",
                         text=state.values, textposition="top center", line=dict(color="#e6e9ef", shape="hv"),
                         showlegend=False, textfont=dict(size=10, color="#c9d1dc")), 2, 1)
lay = ch.base_layout("", 600); lay.pop("xaxis"); lay.pop("yaxis"); lay["barmode"] = "relative"
lay["legend"].update(y=-0.12, yanchor="top", x=0.5, xanchor="center"); lay["hovermode"] = "x unified"
fig.update_layout(**lay); fig.update_xaxes(gridcolor=ch.GRID_COLOR); fig.update_yaxes(gridcolor=ch.GRID_COLOR)
fig.update_yaxes(range=[-0.5, 7], tickvals=[s.value for s in sc.ChokepointStatus],
                 ticktext=[s.name.replace("_", " ").lower() for s in sc.ChokepointStatus], row=2, col=1)
fig.show()

**What changed:** 28 Feb Hormuz threatened → 2 Mar Qatar LNG halt, attacks confirmed → 16 Mar safe passage.

In [4]:
if os.getenv("OPENROUTER_API_KEY"):
    llm = pp.OpenRouterClient(model=os.getenv("OPENROUTER_MODEL", "meta-llama/llama-3.3-70b-instruct:free"))
    live = pp.ExtractionPipeline(pp.InternetArchiveSource(), pp.BriefExtractor(llm, pp.PromptBuilder(few_shot=[GOLD[slugs[1]]])),
                                 pp.JsonFileRepository()).run(slugs)
    display(pd.DataFrame({f"{r.article.report_date:%d %b} {r.article.report_type.value}":
                          (pp.Evaluator.score(r.brief, GOLD[r.article.slug]) if r.brief else {"failed": r.errors[-1][:120]})
                          for r in live}).T)
else:
    print("No OPENROUTER_API_KEY - skipped.")

No OPENROUTER_API_KEY - skipped.


## Limitations
ISW blocks scripts · scores not yet calibrated · live LLM run needs `OPENROUTER_API_KEY`.